# 05-Human-in-the-Loop

### 目录

-   回顾：图会自己选路了
-   一、AI 不该全权做主的时刻
-   二、interrupt：让图停下来
-   三、Command(resume)：让图继续
-   四、Checkpointer：interrupt 的底座（复习）
-   五、四种经典 HIL 模式
-   补充：用 stream_events 驱动 HIL
-   六、动手跑一下
-   七、常见坑速查
-   八、总结

### 回顾：图会记住状态了

前两课我们学了两样关键能力：

-   第三课 **条件路由**：`Command(update=..., goto=...)` 让节点自己决定去哪
-   第四课 **持久化记忆**：Checkpointer + thread_id 让图能存档、能恢复、能改历史

图会选路、会记忆了，一切都「自动」了。但「自动」有时候恰恰是问题——

今天的主题：怎么让图在关键一步停下来，等人拍板之后再继续？

### 一、AI 不该全权做主的时刻

假设你的客服 Agent 已经能自动处理退款。听起来很美好，直到有一天——

-   用户说「给我退 50000 元」，AI 看了一眼，**直接退了**
-   AI 生成了一封措辞不当的道歉邮件，**直接发了**
-   AI 判断要删除用户数据，**直接删了**

这些操作的共同点：**不可逆、高风险**。AI 可以提建议，但最后一步必须人来确认。

> **类比：经理签字才放行**
> 
> 公司报销流程：员工填单（AI 干活）→ 单子放在经理桌上**等签字**（流程暂停）→ 经理签了字（人拍板）→ 财务打款（流程继续）。
> 
> 重点是：等签字期间，**单子不会丢**——它就躺在桌上，经理下周再签也行。

这就是 **Human-in-the-Loop（HIL）**：在自动化流程中插入人工决策点。LangGraph 用两个原语实现它：

<figure style="text-align:center;">
  <img src="../assets/05_hil_concept.png" width="700" />
  <figcaption style="text-align:center; margin-top: 6px;">LangGraph_05_HIL_concept.png</figcaption>
</figure>

### 二、interrupt：让图停下来

`interrupt()` 是一个在**节点内部**调用的函数。执行到它，图立刻暂停，把你传给它的信息抛到外面：



In [1]:
from typing_extensions import TypedDict

from langgraph.types import interrupt


class State(TypedDict):
    amount: int


def human_review(state: State):
    # 执行到这里，图暂停。括号里的内容会被抛给外部调用方
    decision = interrupt(
        {
            "question": "是否批准退款？",
            "amount": state["amount"],
        }
    )
    # ↑ 恢复执行时，interrupt() 的【返回值】就是人传进来的决定
    return {"decision": decision}


调用方拿到的是什么？`invoke` 的返回值里会多一个 `__interrupt__` 字段：



In [2]:
# 行为示意：为了保证整本 notebook 从上到下可执行，这里不直接运行。
# result = graph.invoke(inputs, config)
# print(result["__interrupt__"])
# [Interrupt(value={'question': '是否批准退款？', 'amount': 500}, id='...')]


这时你可以把 `value` 里的信息展示给用户——弹窗、发企业微信、邮件通知审批人，随便你。**图会一直等着，等多久都行。**

> **和 Python 的 input() 有什么区别？**`input()` 会卡死整个进程，服务器场景根本没法用。`interrupt()` 是把状态**存档后退出**——进程可以去服务别的请求，甚至重启机器，下周再恢复这个流程都没问题。

### 三、Command(resume)：让图继续

人做完决定后，用上一课认识的老朋友 `Command` 唤醒图——这次用的是它的第三个参数 `resume`：



In [3]:
from langgraph.types import Command

# 行为示意：为了保证整本 notebook 从上到下可执行，这里不直接运行。
# result = graph.invoke(Command(resume="approve"), config)


发生了什么：

1.  图从存档点恢复，**重新执行**被打断的节点
2.  这次执行到 `interrupt()` 时不再暂停——它直接**返回** `resume` 传入的值（这里是 `"approve"`）
3.  节点继续往下跑，图正常流转到下一站

> **关键细节：节点会从头重新执行！**
> 
> 恢复时**不是**从 `interrupt()` 那一行接着跑，而是**整个节点函数从第一行重跑一遍**（实测验证）。所以：
> 
> • `interrupt()` **之前**的代码会执行两次——别把「扣款」「发邮件」这种副作用写在它前面
> 
> • 最佳实践：让审批节点**只做审批**，副作用放到后面的节点里

至此，HIL 的完整闭环：



In [4]:
# 行为示意：为了保证整本 notebook 从上到下可执行，这里不直接运行。
# ① 第一次调用：跑到 interrupt 暂停
# result = graph.invoke({"amount": 500, ...}, config)
# ② 把 result["__interrupt__"] 展示给人，等决定（几秒或几天）
# ③ 人决定后，恢复执行
# result = graph.invoke(Command(resume="approve"), config)


> **官方提醒：** 在 `invoke`、`stream` 或 `stream_events` 的输入里，**只有 `Command(resume=...)` 是合法的**。不要把 `Command(update=...)` 或 `Command(goto=...)` 当作输入来继续多轮对话。  
> **注意区分：** 节点函数里返回 `Command(update=..., goto=...)` 是完全正确的做法（如上面的“批准/拒绝”模式），它只在“作为输入传给 invoke/stream”时才是错的。

> **❓ 检查理解 ①**
> 
> 节点函数里 `interrupt()` 的**返回值**是什么？
> 
> -   A. 暂停时抛给外部的那个字典
> -   B. 恢复执行时，`Command(resume=...)` 传入的值
> -   C. None，它只负责暂停，不返回东西

> **✅ 答案：B**
> 
> interrupt 是一进一出的通道：括号里的参数【抛出去】给人看，resume 的值【传回来】作为返回值。一个函数完成双向通信。

### 四、Checkpointer：interrupt 的底座（复习）

`interrupt` 不是把进程挂起，而是把当前 State **存档后退出**。之所以能“等多久都行”，靠的是第四课的 Checkpointer + `thread_id`。这里只强调两个关键点：

1.  **必须挂 checkpointer：** 不挂的话第一次 invoke 能拿到 `__interrupt__`，但 resume 时会报 `RuntimeError: Cannot use Command(resume=...) without checkpointer`。
2.  **恢复必须用同一个 thread_id：**`thread_id` 就是“存档位编号”，换了等于换一个存档。

In [5]:
# InMemorySaver + thread_id（示意）
# from langgraph.checkpoint.memory import InMemorySaver
#
# graph = builder.compile(checkpointer=InMemorySaver())  # ① 编译时挂载
# config = {"configurable": {"thread_id": "refund-001"}}        # ② 指定存档位


> **详细复习：** Checkpointer 的完整机制、持久化方案对比（InMemorySaver / SqliteSaver / PostgresSaver）已在上一课 `04_Checkpointer与State管理.ipynb` 中讲解。本课所有示例默认使用 `InMemorySaver`，仅适合开发调试。

### 五、四种经典 HIL 模式

**模式 1：批准 / 拒绝（Approve or Reject）**

最常见：高风险操作前停一下，人批准走 A，拒绝走 B。配合上一课的 `Command(goto=...)`：



In [6]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.types import interrupt, Command


class GateState(TypedDict):
    amount: int
    approved: bool


def approval_gate(state: GateState) -> Command[Literal["执行退款", "婉拒用户"]]:
    decision = interrupt({"question": "批准这笔退款吗？", "amount": state["amount"]})
    if decision == "approve":
        return Command(update={"approved": True}, goto=["执行退款"])
    return Command(update={"approved": False}, goto=["婉拒用户"])


**模式 2：人工编辑（Edit State）**

人不止说「行/不行」，还能**直接改 State**再放行。比如审批人觉得退 500 太多，改成 300：



In [7]:
# 行为示意：这段会在后面的 Demo 1 里真正跑通。
# graph_hil.update_state(config_hil, {"amount": 300})
# result = graph_hil.invoke(Command(resume="approve"), config_hil)


`update_state` 就是上一课学的「修改存档」，规则也一样：普通字段覆盖，带 Reducer 的字段合并。

**模式 3：补充信息（Ask Human）**

AI 干到一半发现缺信息，停下来问人。`resume` 传回的就是人的回答：



In [8]:
from typing_extensions import TypedDict

from langgraph.types import interrupt


class InfoState(TypedDict):
    order_id: str


def collect_info(state: InfoState):
    if not state.get("order_id"):
        order_id = interrupt({"question": "请提供订单号"})
        return {"order_id": order_id}
    return {}


**模式 4：审核循环（Review Loop）**

AI 生成 → 人审 → 不满意打回重做 → 再审，直到通过。这是模式 1 + 上一课自循环的组合：

<figure style="text-align:center;">
  <img src="../assets/05_review_loop.png" width="700" />
  <figcaption style="text-align:center; margin-top: 6px;">LangGraph_05_review_loop.png</figcaption>
</figure>

> **❓ 检查理解 ②**
> 
> 审批人想把退款金额从 500 改成 300 再放行，最合适的做法是？
> 
> -   A. `Command(resume="300")`，在节点里解析这个字符串
> -   B. 暂停期间用 `graph.update_state(config, {"amount": 300})`，再 resume
> -   C. 重新 invoke 一个全新的输入 `{"amount": 300}`

> **✅ 答案：B**
> 
> update_state 就是为「人工编辑」设计的——直接改存档里的 State，后续节点看到的就是修改后的值。resume 只负责传「决定」，改数据交给 update_state，职责清晰。

### 补充：用 stream_events 驱动 HIL（官方推荐）

除了 `invoke`，LangGraph 官方更推荐用 `stream_events(version="v3")` 来驱动可能触发 interrupt 的图。它的好处是：interrupt 信息会通过 `stream.interrupts` 暴露出来，而不是藏在最终结果里，便于前端/UI 实时展示“等待人工输入”的状态。



In [9]:
from langgraph.types import Command

# stream_events 驱动 HIL（示意）
# 注意：需要先有一个会触发 interrupt 的 graph（见后面的 Demo 1）。
#
# config = {"configurable": {"thread_id": "stream-demo"}}
# stream = graph_hil.stream_events({"amount": 500, ...}, config=config, version="v3")
# final = stream.output
# if stream.interrupted:
#     print("需要人工决策：", stream.interrupts)
# resumed = graph_hil.stream_events(Command(resume="approve"), config=config, version="v3")
# final = resumed.output


和 `invoke` 的核心区别：

-   `stream.interrupted` 直接告诉你是否暂停了；
-   `stream.interrupts` 给出 interrupt 抛出的值；
-   适合前端/UI 做“等待审批”提示，不需要去翻 `result["__interrupt__"]`。

> **注意：**`stream_events(version="v3")` 目前仍是实验性 API，未来可能调整。本课主示例仍使用更稳定的 `invoke`，但生产/前端对接建议关注 v3。

### 六、动手跑一下

**Demo 1：退款审批流**

完整的「暂停 → 查看 → 恢复」闭环：



In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

# ---- State ----
class State(TypedDict):
    order: str
    amount: int
    decision: str
    result: str

# ---- 节点 ----
def prepare_refund(state: State):
    print(f"[准备] 订单 {state['order']}，金额 {state['amount']} 元")
    return {}


def human_approve(state: State):
    # 图在这里暂停，把审批信息抛给外部
    decision = interrupt(
        {
            "question": "是否批准退款？",
            "order": state["order"],
            "amount": state["amount"],
        }
    )
    return {"decision": decision}


def execute_refund(state: State):
    if state["decision"] == "approve":
        return {"result": f"已退款 {state['amount']} 元"}
    return {"result": "退款被拒绝"}

# ---- 构建图（注意 checkpointer！）----
builder = StateGraph(State)
builder.add_node("准备退款", prepare_refund)
builder.add_node("人工审批", human_approve)
builder.add_node("执行退款", execute_refund)
builder.add_edge(START, "准备退款")
builder.add_edge("准备退款", "人工审批")
builder.add_edge("人工审批", "执行退款")
builder.add_edge("执行退款", END)

graph_hil = builder.compile(checkpointer=InMemorySaver())
config_hil = {"configurable": {"thread_id": "refund-001"}}

# ---- ① 第一次运行：跑到 interrupt 停下 ----
result = graph_hil.invoke(
    {"order": "A1001", "amount": 500, "decision": "", "result": ""},
    config_hil,
)
print("暂停时抛出:", result["__interrupt__"][0].value)
print("暂停在节点:", graph_hil.get_state(config_hil).next)

# ---- ② 人看完信息，批准了 → 恢复执行 ----
result = graph_hil.invoke(Command(resume="approve"), config_hil)
print("最终结果:", result["result"])

[准备] 订单 A1001，金额 500 元
暂停时抛出: {'question': '是否批准退款？', 'order': 'A1001', 'amount': 500}
暂停在节点: ('人工审批',)
最终结果: 已退款 500 元



运行后你应该看到（实测输出）：

```text
[准备] 订单 A1001，金额 500 元
暂停时抛出: {'question': '是否批准退款？', 'order': 'A1001', 'amount': 500}
暂停在节点: ('人工审批',)
最终结果: 已退款 500 元
```

**关键观察：**

-   两次 `invoke` 之间，图「失忆地」躺在存档里——这中间你可以做任何事，等任何久
-   `get_state(config).next` 能看到暂停在哪个节点，适合做管理后台
-   把 `resume="approve"` 改成 `"reject"` 再跑一次，走的就是拒绝分支

**Demo 2：审核循环（打回重写）**

模式 4 的完整实现：AI 写稿 → 人审 → 不满意打回 → 修改后再审。保存为 `demo_review_loop.py`：



In [11]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    text: str
    status: str

# 审核节点：interrupt + Command(goto) 二合一
def human_review(state: State) -> Command[Literal["发布", "重写"]]:
    decision = interrupt({"draft": state["text"], "question": "通过还是打回？"})
    if decision == "ok":
        return Command(update={"status": "approved"}, goto=["发布"])
    return Command(update={"text": state["text"] + "(修改版)"}, goto=["重写"])


def publish(state: State):
    return {"status": "已发布"}


def rewrite(state: State):
    print(f"  重写后再次送审: {state['text']}")
    return {}

builder = StateGraph(State)
builder.add_node("人审", human_review)
builder.add_node("发布", publish)
builder.add_node("重写", rewrite)
builder.add_edge(START, "人审")
builder.add_edge("重写", "人审")   # 重写完回到审核——循环！
builder.add_edge("发布", END)

graph_review = builder.compile(checkpointer=InMemorySaver())
config_review = {"configurable": {"thread_id": "review-001"}}

graph_review.invoke({"text": "初稿", "status": ""}, config_review)
print("第一次审核 → 打回")
graph_review.invoke(Command(resume="no"), config_review)  # 打回 → 重写 → 自动再次暂停
print("第二次审核 → 通过")
result = graph_review.invoke(Command(resume="ok"), config_review)
print(f"最终: {result['status']} | {result['text']}")

第一次审核 → 打回
  重写后再次送审: 初稿(修改版)
第二次审核 → 通过
最终: 已发布 | 初稿(修改版)



运行后（实测输出）：

```text
第一次审核 → 打回
  重写后再次送审: 初稿(修改版)
第二次审核 → 通过
最终: 已发布 | 初稿(修改版)
```

注意第二次 `resume="no"` 之后：图走完「重写」又回到「人审」，**自动再次暂停**——一个 invoke 里发生了「恢复 → 流转 → 再暂停」，这就是 HIL 循环的节奏。

**试一试：人工编辑**

在 Demo 1 的两次 invoke 之间插入一行，把金额改成 300 再批准：



In [12]:
# Demo 1 已经构建好了 graph_hil，这里演示“人工编辑（Edit State）”的正确顺序：
# 先跑到 interrupt 暂停 → 再 update_state 改存档 → 最后 resume 继续跑

config_edit = {"configurable": {"thread_id": "refund-edit-001"}}

# ① 先触发一次 interrupt（暂停）
result = graph_hil.invoke(
    {"order": "A1002", "amount": 500, "decision": "", "result": ""},
    config_edit,
)
print("暂停时抛出:", result["__interrupt__"][0].value)

# ② 暂停期间，人工把退款金额从 500 改成 300
graph_hil.update_state(config_edit, {"amount": 300})

# ③ 再恢复执行——后续节点读到的 amount 就是 300
result = graph_hil.invoke(Command(resume="approve"), config_edit)
print("最终结果:", result["result"])

[准备] 订单 A1002，金额 500 元
暂停时抛出: {'question': '是否批准退款？', 'order': 'A1002', 'amount': 500}
最终结果: 已退款 300 元



> **先猜猜输出是多少，再点开**
> 
> 实测输出：`最终结果: 已退款 300 元`——「执行退款」节点读到的已经是改过的 State。这就是模式 2 人工编辑的威力。

### 七、常见坑速查

| 症状  | 原因  | 解法  |
| --- | --- | --- |
| 用 Command(update=...) 或 Command(goto=...) 作为输入继续运行 | 只有 Command(resume=...) 是设计给 invoke/stream/stream_events 作为输入的；update/goto/graph 参数只能在节点函数里返回 | 恢复执行只传 Command(resume=...)；继续多轮对话传普通输入 dict |
| RuntimeError: Cannot use Command(resume=...) without checkpointer | 编译时没挂 checkpointer。第一次 invoke 不报错，resume 时才炸（隐蔽！） | compile(checkpointer=InMemorySaver()) |
| resume 后图「从零开始」跑了新流程 | resume 时用了不同的 thread_id，或传了普通输入而非 Command(resume=...) | 恢复必须：同一个 thread_id + Command(resume=...) |
| 扣款 / 发邮件执行了两次 | 副作用写在了 interrupt() 之前——恢复时整个节点从头重跑 | 副作用移到 interrupt 之后，或拆到下一个节点 |
| 重启进程后存档全丢了 | InMemorySaver 只存内存 | 生产环境用 SqliteSaver / PostgresSaver（第四课已详解，或参考官方持久化集成文档） |
| 一个节点里多个 interrupt，恢复后对不上号 | 多次 resume 按 interrupt 的调用顺序匹配，节点重跑时顺序必须稳定 | 一个节点尽量只放一个 interrupt；多个就保证调用顺序固定不变 |

### 八、总结

| 概念  | 一句话解释 | 类比  |
| --- | --- | --- |
| interrupt(信息) | 节点内暂停，把信息抛给外部；恢复时返回人的决定 | 单子放经理桌上 |
| Command(resume=决定) | 用同一 thread_id 唤醒图，决定值进入 interrupt 的返回值 | 经理签字放行 |
| Checkpointer | 自动存档系统，暂停/恢复的底层支撑 | 游戏存档 |
| thread_id | 存档槽位编号，恢复时必须对上 | 存档槽位 |
| update_state | 暂停期间人工直接修改 State | 经理改单子再签 |
| 节点重跑 | 恢复时整个节点从头执行，副作用别放 interrupt 前 | 读档回到房间门口，不是 BOSS 面前 |

> **核心记忆**
> 
> **interrupt 抛出去，resume 传回来，Checkpointer 兜住一切。**
> 
> 四种模式：批准/拒绝、人工编辑（update_state）、补充信息、审核循环。它们都是这三件套的组合拳。

> **下节课预告**
> 
> 到目前为止，节点里的「判断」都是手写规则（`if "退货" in query`）。第六课**接入 LLM**——把大模型装进节点，分类、生成、决策真正交给 AI。学完它，本课的审批流就能升级成「AI 起草 + 人终审」的完整系统。

📖 参考：

-   [LangGraph 官方文档 - Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
-   [Persistence 文档](https://docs.langchain.com/oss/python/langgraph/persistence)